# Notebook 3 — Pipeline de Monitoramento e Avaliação de SLOs

## Aula 6: Monitoramento Contínuo e Data SLOs em Produção

### Objetivos

1. Executar o **pipeline completo de monitoramento** nos dados de produção.
2. Verificar os 3 SLOs: acurácia (Snippet 1), drift KS (Snippet 2), qualidade (Snippet 3).
3. Comparar métricas de referência vs produção para quantificar degradação.
4. Analisar **Error Budget** e gerar relatório de monitoramento.

### Conexão com o Documento 04

> *"Para implementar o monitoramento contínuo na prática, é necessário reunir
> dados de referência e métricas em tempo real. Uma arquitetura típica de
> monitoramento de ML inclui: log de previsões e rótulos reais em produção,
> verificação periódica de qualidade e integridade dos dados, além de
> detecção automática de drift."*
> — DOCUMENTO_AULA_6.md, seção 'Boas Práticas, Ferramentas e Estado da Arte'

### Vídeos Relacionados

- **Vídeo 6.3**: Pipeline de monitoramento com SLOs em Python
- **Vídeo 6.4**: Ferramentas de MLOps para monitoramento contínuo

In [ ]:
# Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Adicionar diretório pai ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor
from src.model import SLOMonitor, MonitoringPipeline
from src.evaluation import ModelEvaluator, calculate_brier_score
from src.utils import load_model, load_metrics, save_metrics, setup_logging

setup_logging()

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

%matplotlib inline

## 1. Carregar Modelo e Dados

Carregamos o modelo treinado no Notebook 2 e preparamos os dados
de referência e produção para o pipeline de monitoramento.

In [ ]:
# Carregar modelo treinado
model = load_model("../outputs/models/credit_model.joblib")
print(f"Modelo carregado: {type(model).__name__}")

# Carregar métricas baseline
baseline = load_metrics("../outputs/logs/baseline_metrics.json")
print(f"\nMétricas baseline:")
for k, v in baseline["metrics"].items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Carregar e preparar dados
preprocessor = DataPreprocessor(random_state=42)
data = preprocessor.load_data()

# Separar referência e produção
df_ref, df_prod = preprocessor.split_reference_production(data)

# Limpar e preparar features
df_ref_clean = preprocessor.clean_data(df_ref)
df_prod_clean = preprocessor.clean_data(df_prod)

X_ref, y_ref = preprocessor.prepare_features(df_ref_clean)
X_prod, y_prod = preprocessor.prepare_features(df_prod_clean)

## 2. Previsões em Produção

Aplicamos o modelo treinado nos dados de produção.
Conforme o **Snippet 1** do DOCUMENTO_AULA_6.md:

```python
accuracy = accuracy_score(y_true, y_pred)
SLO = 0.85
if accuracy < SLO:
    alert(f"Acurácia atual {accuracy:.1%} abaixo do SLO de 85%!")
```

In [ ]:
# Gerar previsões nos dados de produção
y_pred_prod = model.predict(X_prod)
y_proba_prod = model.predict_proba(X_prod)[:, 1]

# Gerar previsões nos dados de referência (para comparação)
y_pred_ref = model.predict(X_ref)
y_proba_ref = model.predict_proba(X_ref)[:, 1]

print(f"Previsões geradas: {len(y_pred_prod)} amostras de produção")

## 3. Pipeline de Monitoramento Completo

Executamos o `MonitoringPipeline` que integra os 3 checks de SLO
definidos nos Snippets 1-3 do DOCUMENTO_AULA_6.md:

1. **Check de Acurácia** (Snippet 1): accuracy ≥ 85%
2. **Check de Drift KS** (Snippet 2): p-value ≥ 0.05 por feature
3. **Check de Qualidade** (Snippet 3): missing rate ≤ 1%

> *"Nas videoaulas, você verá esses e outros exemplos de checks de qualidade,
> além da integração com ferramentas especializadas."*
> — DOCUMENTO_AULA_6.md, seção 'Hands On'

In [ ]:
# Configurar SLO Monitor conforme thresholds do doc 04
slo_monitor = SLOMonitor(
    accuracy_slo=0.85,
    drift_p_value_threshold=0.05,
    missing_rate_slo=0.01,
    psi_threshold=0.25,
)

# Configurar pipeline com as features mais relevantes
pipeline = MonitoringPipeline(
    slo_monitor=slo_monitor,
    features_to_monitor=[
        "idade", "renda_mensal", "score_credito",
        "tempo_emprego", "valor_emprestimo",
    ],
)

# Executar pipeline completo
# Usa dados de produção SEM limpar (df_prod original) para detectar missings
report = pipeline.run_full_check(
    y_true=y_prod.values,
    y_pred=y_pred_prod,
    df_reference=df_ref,
    df_production=df_prod,
)

print(f"\n{'=' * 60}")
print(f"RELATÓRIO DE MONITORAMENTO")
print(f"{'=' * 60}")
print(f"Status: {report.overall_status}")
print(f"Checks OK:  {report.passed_count}/{len(report.checks)}")
print(f"Checks NOK: {report.failed_count}/{len(report.checks)}")
print(f"\n{report.summary}")

In [ ]:
# Detalhar resultados de cada check
print("\nDetalhes dos Checks:")
print("-" * 80)

checks_data = []
for check in report.checks:
    status = "✅" if check.passed else "❌"
    checks_data.append({
        "Status": status,
        "Check": check.name,
        "Valor": round(check.metric_value, 4),
        "Threshold": check.threshold,
        "Mensagem": check.message[:80],
    })

checks_df = pd.DataFrame(checks_data)
checks_df

## 4. Alertas Gerados

Conforme a seção 'Boas Práticas' do DOCUMENTO_AULA_6.md:
> *"Define-se regras de alerta (por exemplo, enviar uma notificação
> no Slack ou Teams se um SLO não é atendido)."*

In [ ]:
# Listar alertas
alerts = pipeline.get_alerts()

if alerts:
    print(f"⚠️  {len(alerts)} ALERTAS gerados:\n")
    for i, alert_msg in enumerate(alerts, 1):
        print(f"  [{i}] {alert_msg}")
else:
    print("✅ Nenhum alerta — todos os SLOs atendidos.")

## 5. Comparação de Métricas: Referência vs Produção

Quantificamos a degradação de performance do modelo quando
aplicado aos dados de produção com drift.

Conforme o DOCUMENTO_AULA_6.md:
> *"Modelos de ML são altamente sensíveis a mudanças no ambiente
> e nos dados. Mesmo alterações sutis no perfil dos dados de entrada
> podem degradar significativamente sua performance."*

In [ ]:
# Calcular métricas em referência e produção
evaluator = ModelEvaluator(figures_dir="../outputs/figures")

metrics_ref = evaluator.calculate_metrics(y_ref.values, y_pred_ref, y_proba_ref)
metrics_prod = evaluator.calculate_metrics(y_prod.values, y_pred_prod, y_proba_prod)

# Comparar
comparison = evaluator.compare_metrics(metrics_ref, metrics_prod)

print("Comparação de Métricas: Referência vs Produção")
print("=" * 60)
for metric, values in comparison.items():
    delta = values["delta"]
    trend = "↗️" if delta > 0.01 else "↘️" if delta < -0.01 else "→"
    print(
        f"  {metric:15s}: "
        f"Ref={values['reference']:.4f} → "
        f"Prod={values['production']:.4f} "
        f"(Δ={delta:+.4f}) {trend}"
    )

In [ ]:
# Gráfico comparativo
fig = evaluator.plot_metrics_comparison(
    metrics_ref, metrics_prod,
    title="Degradação de Performance: Referência vs Produção\n"
          "(Conforme seção 'Por Que Monitorar Modelos de ML' do doc 04)",
    filename="metrics_comparison_ref_vs_prod.png",
)
plt.show()

## 6. Brier Score — Calibração sob Drift

Conforme **Ovadia et al. (2019)**, citado no DOCUMENTO_AULA_6.md:
> *"Tanto a acurácia quanto a calibração dos modelos costumam se
> deteriorar sob dataset shift — ou seja, métodos modernos sofrem
> redução de acerto e ficam super ou sub-confiantes quando expostos
> a dados cuja distribuição diverge daquela original."*

In [ ]:
# Brier Score: referência vs produção
brier_ref = calculate_brier_score(y_ref.values, y_proba_ref)
brier_prod = calculate_brier_score(y_prod.values, y_proba_prod)

print(f"Brier Score (Referência): {brier_ref:.4f}")
print(f"Brier Score (Produção):   {brier_prod:.4f}")
print(f"Degradação:               {brier_prod - brier_ref:+.4f}")
print(f"\n{'⚠️  Calibração piorou sob drift!' if brier_prod > brier_ref else '✅ Calibração OK'}")

In [ ]:
# Curvas de calibração comparativas
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

fig, ax = plt.subplots(figsize=(8, 6))

# Referência
frac_ref, mean_ref = calibration_curve(y_ref.values, y_proba_ref, n_bins=10)
ax.plot(mean_ref, frac_ref, marker="o", lw=2, color="steelblue",
        label=f"Referência (Brier={brier_ref:.4f})")

# Produção
frac_prod, mean_prod = calibration_curve(y_prod.values, y_proba_prod, n_bins=10)
ax.plot(mean_prod, frac_prod, marker="s", lw=2, color="coral",
        label=f"Produção (Brier={brier_prod:.4f})")

ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfeitamente calibrado")
ax.set_xlabel("Probabilidade prevista média")
ax.set_ylabel("Fração de positivos")
ax.set_title(
    "Calibração sob Dataset Shift\n"
    "(Ovadia et al., 2019 — citado no doc 04)"
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/figures/calibration_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Matriz de Confusão — Produção

Analisamos os erros do modelo nos dados de produção com drift.

In [ ]:
# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Referência
from sklearn.metrics import confusion_matrix
cm_ref = confusion_matrix(y_ref.values, y_pred_ref)
sns.heatmap(cm_ref, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Adim.", "Inadim."], yticklabels=["Adim.", "Inadim."])
axes[0].set_title("Matriz de Confusão — Referência")
axes[0].set_xlabel("Previsão")
axes[0].set_ylabel("Real")

# Produção
cm_prod = confusion_matrix(y_prod.values, y_pred_prod)
sns.heatmap(cm_prod, annot=True, fmt="d", cmap="Oranges", ax=axes[1],
            xticklabels=["Adim.", "Inadim."], yticklabels=["Adim.", "Inadim."])
axes[1].set_title("Matriz de Confusão — Produção (com drift)")
axes[1].set_xlabel("Previsão")
axes[1].set_ylabel("Real")

plt.suptitle(
    "Impacto do Drift na Performance do Modelo",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig("../outputs/figures/confusion_matrices_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Error Budget Analysis

Conforme o Vídeo 6.2 — Error Budget é a margem de tolerância
entre o desempenho atual e o SLO.

$$
\text{Error Budget} = \text{Métrica Atual} - \text{SLO}
$$

Se o Error Budget se esgota (< 0), é necessário tomar ação
corretiva (retrain, investigação, rollback).

In [ ]:
# Error Budget: acurácia em produção vs SLO
from sklearn.metrics import accuracy_score

accuracy_ref = accuracy_score(y_ref.values, y_pred_ref)
accuracy_prod = accuracy_score(y_prod.values, y_pred_prod)
slo_accuracy = 0.85

error_budget_ref = accuracy_ref - slo_accuracy
error_budget_prod = accuracy_prod - slo_accuracy

print("Error Budget Analysis")
print("=" * 50)
print(f"SLO de Acurácia:        {slo_accuracy:.0%}")
print(f"Acurácia Referência:    {accuracy_ref:.1%}  (budget: {error_budget_ref:+.1%})")
print(f"Acurácia Produção:      {accuracy_prod:.1%}  (budget: {error_budget_prod:+.1%})")
print()
if error_budget_prod < 0:
    print("❌ ERROR BUDGET ESGOTADO em produção!")
    print("   Ação necessária: investigar drift, considerar retrain.")
else:
    print(f"✅ Error Budget disponível: {error_budget_prod:.1%}")

In [ ]:
# Visualização do Error Budget
fig, ax = plt.subplots(figsize=(10, 5))

categories = ["Referência", "Produção"]
accuracies = [accuracy_ref, accuracy_prod]
colors = [
    "green" if a >= slo_accuracy else "red" for a in accuracies
]

bars = ax.bar(categories, accuracies, color=colors, alpha=0.8, width=0.5)
ax.axhline(y=slo_accuracy, color="red", linestyle="--", linewidth=2,
           label=f"SLO = {slo_accuracy:.0%}")

# Anotar error budget
for bar, budget in zip(bars, [error_budget_ref, error_budget_prod]):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f"Budget: {budget:+.1%}",
        ha="center", fontsize=12, fontweight="bold",
    )

ax.set_ylabel("Acurácia")
ax.set_title(
    "Error Budget — Acurácia vs SLO\n"
    "(Conceito discutido no Vídeo 6.2)"
)
ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig("../outputs/figures/error_budget.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Relatório Final e Salvamento

Geramos o relatório completo e salvamos os resultados do monitoramento.

Conforme seção 'Mercado, Cases e Tendências' do DOCUMENTO_AULA_6.md:
> *"O caso Zillow ilustra que erros de modelos em produção podem
> gerar prejuízos financeiros e danos à reputação. Empresas que
> investem em monitoramento proativo colhem resultados positivos."*

In [ ]:
# Relatório de classificação — produção
report_text = evaluator.generate_classification_report(
    y_prod.values, y_pred_prod
)
print("Relatório de Classificação — Produção\n")
print(report_text)

In [ ]:
# Salvar relatório de monitoramento
monitoring_results = {
    "status": report.overall_status,
    "metrics_reference": metrics_ref,
    "metrics_production": metrics_prod,
    "brier_reference": brier_ref,
    "brier_production": brier_prod,
    "error_budget_accuracy": error_budget_prod,
    "total_checks": len(report.checks),
    "passed_checks": report.passed_count,
    "failed_checks": report.failed_count,
    "alerts": alerts,
    "report_detail": report.to_dict(),
}

save_metrics(monitoring_results, "../outputs/logs/monitoring_report.json")
print("Relatório de monitoramento salvo em outputs/logs/monitoring_report.json")

## Resumo

### O que aprendemos neste notebook:

1. **Pipeline de Monitoramento** integrado verificando 3 SLOs:
   acurácia (Snippet 1), drift KS (Snippet 2) e qualidade (Snippet 3).

2. **Degradação confirmada**: o modelo perde performance nos dados
   de produção com drift, conforme previsto pela teoria.

3. **Brier Score** piorou sob drift, confirmando Ovadia et al. (2019):
   *"modelo ficam super ou sub-confiantes"* sob dataset shift.

4. **Error Budget** consumido ou esgotado em produção, sinalizando
   necessidade de ação corretiva.

5. **Alertas automáticos** gerados para features com drift e
   colunas com qualidade abaixo do SLO.

### Aprendizados-chave da Aula 6

- Monitorar modelos não é opcional: *"modelos de ML não são soluções
  configure e esqueça"* (DOCUMENTO_AULA_6.md).
- Data SLOs definem metas claras de qualidade (Breck et al., 2017).
- Ferramentas como Evidently, Great Expectations e Prometheus/Grafana
  facilitam o monitoramento em escala (Vídeo 6.4).

### Referências

- Breck, E. et al. (2017). The ML Test Score. IEEE BigData.
- Gama, J. et al. (2014). A survey on concept drift. ACM Computing Surveys.
- Ovadia, Y. et al. (2019). Can You Trust Your Model's Uncertainty? NeurIPS.
- Sculley, D. et al. (2015). Hidden Technical Debt in ML Systems. NIPS.